# Pipeline 04Gc: global Vision → text (G2a, per feature)

For each `(model, feature)`: the LLM receives the feature's **global plot as an image**
(EBM shape plot / XGB SHAP dependence plot) and writes a
`[EFFECT] / [IMPORTANCE] / [RECOMMENDATION]` description. The importance rank can't be
read off the plot, so it is stated in the user message (the one datum vision lacks vs.
JSON). System prompt = shared core + vision handover ({{MODEL}} + {{ARTIFACT}} filled,
cached). Output: `results/global/vision_{model}_{feature}.json`. Resumable.

In [1]:
from __future__ import annotations

import sys, time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from utils import (
    EXPLANATIONS_DIR, RESULTS_DIR, PROMPTS_DIR, GLOBAL_RESULTS_SUBDIR,
    list_global_features, feature_importance_map, shape_plot_path,
    assemble_global_system_prompt, build_global_record, run_resumable_global_generation,
)
from utils.llm import ask_with_images, DEFAULT_MODEL, MAX_TOKENS_GENERATION, strip_scratchpad

LOSS_KEY   = 'poisson_log'
MODEL      = DEFAULT_MODEL
MAX_TOKENS = MAX_TOKENS_GENERATION
XAI_MODELS = ['xgb', 'ebm']

PLOTS_DIR = EXPLANATIONS_DIR / 'plots' / 'global'
OUT_DIR   = RESULTS_DIR / GLOBAL_RESULTS_SUBDIR
OUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURES   = list_global_features('ebm', explanations_dir=EXPLANATIONS_DIR)
N_FEATURES = len(FEATURES)
RANKS      = {m: feature_importance_map(m, explanations_dir=EXPLANATIONS_DIR) for m in XAI_MODELS}

print(f'LLM model: {MODEL}')
print(f'Features:  {N_FEATURES} x {len(XAI_MODELS)} models = {N_FEATURES * len(XAI_MODELS)} calls')
print(f'Plots in:  {PLOTS_DIR}')
print(f'Output:    {OUT_DIR}')

LLM model: claude-sonnet-4-6
Features:  9 x 2 models = 18 calls
Plots in:  /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation-XAI-Stahl-ss26/explanations/plots/global
Output:    /Users/anton/Desktop/SoSe26/Belegarbeit/Implementation-XAI-Stahl-ss26/results/global


In [2]:
# Vision system prompt per model: shared core + vision handover; {{MODEL}} and
# {{ARTIFACT}} (EBM shape plot / XGB SHAP dependence plot) filled. Cached.
SYSTEM = {m: assemble_global_system_prompt('vision', m, prompts_dir=PROMPTS_DIR) for m in XAI_MODELS}
print(SYSTEM['xgb'][:500], '\n...')

You are an expert in explainable AI (XAI). You describe, for staff of a bike rental
company with no technical background, how **one single feature** influences the
predicted demand, always for the feature named in the user message.

## DOMAIN CONTEXT

The Capital Bikeshare system in Washington D.C. rents bikes by the hour. A **XGBoost**
model predicts how many bikes (`cnt`) are rented in a given hour. every statement you
make is about this XGBoost model. It was trained with Poisson deviance loss 
...


In [3]:
# Resume/persistence via utils.run_resumable_global_generation. The plot image rides
# in the user message (not cached); only the system prompt caches (per model).

def generate_vision(model_name, feature, gen_idx):
    plot_path = shape_plot_path(model_name, feature, plots_dir=PLOTS_DIR)
    rank = RANKS[model_name][feature]['rank']
    user = (
        f'Describe the global effect of the feature "{feature}" on hourly bike demand.\n'
        f'Its global importance rank is {rank} of {N_FEATURES}.\n'
        f"The attached image is this feature's global plot."
    )
    t0 = time.time()
    try:
        response = ask_with_images(user, [plot_path], system=SYSTEM[model_name],
                                   model=MODEL, max_tokens=MAX_TOKENS, cache_system=True)
    except Exception as e:
        print(f'  [ERROR] {model_name} {feature}: {type(e).__name__}: {e} -> skip')
        return None
    elapsed = time.time() - t0

    text  = strip_scratchpad(response['content'][0]['text'])
    usage = response.get('usage', {})
    record = build_global_record(
        form='vision', model_name=model_name, feature=feature, explanation=text,
        usage=usage, llm_model=MODEL, loss_key=LOSS_KEY, elapsed_s=round(elapsed, 2),
        extra={'plot_file': plot_path.name},
    )
    u = record['usage']
    print(f"  {model_name.upper()} {feature:11} plot={plot_path.name:24} "
          f"in={u['input_tokens']} out={u['output_tokens']} "
          f"cache={u.get('cache_read_input_tokens', 0)} t={elapsed:.1f}s")
    return record


results = run_resumable_global_generation(
    form='vision', model_names=XAI_MODELS, features=FEATURES,
    out_dir=OUT_DIR, generate=generate_vision,
)

totals = {k: sum(r['usage'].get(k, 0) for r in results)
          for k in ('input_tokens', 'output_tokens', 'cache_read_input_tokens')}
print(f"\nTotal: {totals}  ({len(results)} descriptions)")

  XGB weekday     plot=xgb_dependence_weekday.png in=1601 out=1698 cache=0 t=33.6s

Total: {'input_tokens': 27759, 'output_tokens': 13822, 'cache_read_input_tokens': 0}  (18 descriptions)
